# DATA ANALYSIS PROJECT: CUSTOMER CHURN ANALYSIS.
## Project Overview

.Objective: Analyze customer data to identify churn patterns, key drives (e.g, contract type, service, demographics), and recommend retention.
.Data Source: Public CSV from IBM(Teleco customer churn dataset on github kaggle)
.Scope and Assumptions: Focus on descriptive/Diagnostic analysis. Assumes data from US-like telecom market; 'TotalCharges' has bklanks to handle.
.Tools Used: Python(Pandas for manipulation, Numpy for stats, Matplotlib for visuals.)

.Methodologies Overview:
    .Descriptive Analysis: Summarize distribution(e.g, tenure)
    .Diagonostic Analysis: Explore relationship (e.g, churn vs contract) via correlations and grouping.
    .Exploratory Approach: Visuals-first, iterative to detect biases(e.g, churn imbalances ~26% yes.)
    .Ethical Consideration: Anonymize data; check for demographic biases(e.g, senior citizens.)  

# STEP 1: DATA LOADING.
## Methodology
    .Purpose: import CSV, verify structure, and get an initial overview to get an initial overview to spot loading issues(e.g, type mismatches).
    .Key Principles: Use Pandas for effective reading; parse as needed. inpects shape, columns, types, and samples to understand semantics.
    .Best Practices: Handle potential encoding errors; sample for large files. Add error handling for file access.


## Workflow.
1. Import libraries with aliases for readebility.
2. Load the CSV with error handling(local path.)
3. Initial inspection: Detailed prints for shape, columns, dtypes, head, and describe.

In [ ]:
# Step 1: Import  Necessary dat  libraries with standard aliases for consistency.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

local_path = 'Telco-Customer-Churn.csv'
df = pd.read_csv(local_path, low_memory=False)

# Detailed initial inspection

print("\nData Shape (rows, columns):", df.shape) # cofirms dataset size.
print("\nColumns List:", df.columns.tolist()) # List all column names for reference
print("\nData types:\n", df.dtypes) # show types
print("\nFirst 5 rows (Sample Data):\n", df.head(5)) # preview to spot obvious issues.
print("\nSummary Statistics (including categoricals):\n", df.describe(include='all')) # Mean, count, unique.
      


# Findings and Notes.
. Shape confirms 7043 rows; dytypes reveal 'totalcharges' needs conversion.
. Comments explain purpose. prints are structed for easy debugging.

# STEP 2: DATA CLEANING.
## Methodology.
    .Purpose: fix quality issues like types, missing values, and placeholders to make data analysis-ready.
    .Key Principles: Impute logically(blanks in 'totalcharges' as 0 for new customers); map placeholders('no internet services' to 'no'); drop irrelevant('Customer id' after check).

## Workflow.
    1. Handle missing\blanks with imputation.
    2. Convert types and map values.
    3. Check\remove duplicates and drop unnecessary columns.
    4. Verify with post cleaning stats and optional assertion for data integrity.

In [ ]:
# DATA CLEANING.
#1. Handle missing values in 'TotalCharges' (convert blanks into 'NaN', then Impute)
# First, Coerce to numeric, which turns non-numeric (blanks) to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("\nMissing Values in TotalCharges after coercion:", df['TotalCharges'].isnull().sum())

# Impute NaNs: For new customers (tenur=0), set to 0; else, use monthlycharges * tenure as estimate.
df['TotalCharges'] = np.where(
    df['tenure'] == 0, # conditon for new customer.
    0,                 # value if true
    df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure']) # Impute if false
)
print("Missing Values in TotalCharges after imputation:", df['TotalCharges'].isnull().sum())

#2. convert binary categoricals to 1/0 for easier numerical analysis (e.g, correlations)
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn'] # list of yes/no columns
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.capitalize()
        df[col].fillna('No', inplace=True)

df[binary_cols] = df[binary_cols].replace({'Yes': 1, 'No': 0}).astype(int) # explicit type conversion

# verify dtypes for binaries
print("\nDtypes for binary columns after conversion:\n", df[binary_cols].dtypes) # should be int64
print("nSamples Binary Data (including churn):\n", df[binary_cols].head()) # check 1/0 values

#3. map placeholders in service columns (e.g, 'no internet service' to 'no' for conscistency)
service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df[service_cols] = df[service_cols].replace({'No phone service': 'No', 'No internet service': 'No'})

#4. check and drop duplicates (if any), and drop irrelevant columns 
initial_shape = df.shape
df.drop_duplicates(inplace=True)
print("\nDuplicates Dropped:", initial_shape[0] - df.shape[0])

# Post cleaning validation: Assert no missing values remain key columns
assert df.isnull().sum().sum() == 0, "Warning: Missing still present!"
print("\nCleaned Data shape:", df.shape)
print("\nCleaned Summary Stastistics:\n", df.describe())


# Findings and Notes
.Imputations fixes11 rows; no duplicates; assertions ensure cleanliness
.validation assert prevents silent errors; prints track changes

# STEP 3: EXPLORATORY DATA ANALYSIS.
## Methodology
    .Purpose: Profile variable (univariate), and relationships (bivariate/multivariate) to uncover patterns and churn drivers.
    .Key Principles: Start with distributions; use grouping for categoricals; check correlations.
    .Best Practices: Handle imbalance with normalized crosstabs; limit visuals to keys.

## Workflow
    1. Univariate analysis with stats and plots
    2. Bivariate with crosstabs and correlations.
    3. Multivariate with pairplots/heatmaps.
    4. Generate hypotheses based on outputs.

In [ ]:
# EDA with modular functions for reusability

# Helper function for univariate numerical summary and plot
def univariate_numerical(col, title):
    print(f"\n{col} Description:\n", df[col].describe()) # stats
    plt.figure(figsize=(8, 5)) # figure size for clarity
    sns.histplot(df[col], kde=True, bins=30) # Histogram with kernel density estimate.
    plt.title(title)
    plt.xlabel(col)
    plt.ylabel('frequency')
    plt.show()

univariate_numerical('tenure', 'Tenure Distribution')

# univariate categorical: Value counts and countplot
def univariate_categorical(col, title):
    print(f"\n{col} Value Counts (normalized):\n", df[col].value_counts(normalize=True))
    plt.figure(figsize=(10, 6))
    sns.countplot(x=col, data=df)
    plt.title(title)
    plt.show()

univariate_categorical('Churn', 'Churn Distribution')

# Bivariate: Crosstab with normalization and stacked bar plot
def bivariate_crosstabs(col1, col2, title):
    crosstab = pd.crosstab(df[col1], df[col2], normalize='index')
    print(f"Crosstab {col1} vs {col2}:\n", crosstab)
    crosstab.plot(kind='bar', stacked=True, figsize=(8, 5))
    plt.title(title)
    plt.ylabel('Propotion')
    plt.show()

bivariate_crosstabs('Contract', 'Churn', 'Churn Rate By Contract Type') # high churn in month-to-month

# Overall correlation for numerics
numeric_df = df.select_dtypes(include=['float', 'int']) # filter numerical columns
corr_series = numeric_df.corr()["Churn"].sort_values(ascending=False) # focus on churn correlation
print("\nCorrelation with Churn:\n", corr_series)

# Multivariate: Pairplot with hue for Churn.
plt.figure(figsize=(10, 8))
sns.pairplot(numeric_df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']], hue='Churn', diag_kind='kde')
plt.suptitle('Pairplot of key Numeric by Churn')
plt.show()


Findings and Notes
.Tenure hist shows right skew; crosstab reveals 42% churn for month to month; correlations: tenure(-0.35) strongest negative.
.Why detailed? Functions make code reusable; figsize for better visuals; prints include labels for context.

STEP 4: VISUALIZATION AND INTERPRETATION
Methodology
    .Purpose: Create compelling visuals to communicate insights; intepret actionable recommendations.
    .key Principles: Use appropriate charts (bars for categories, scatter for relationships)

WorkFlow.
    1. Targeted visuals building on EDA.
    2. Heatmap for overview.
    3. Boxplots for distributions.
    3. Summarize interpretations.


In [ ]:
# visualization with customizations

# Visual 1: Barplot for average Churn by internetService
plt.figure(figsize=(8, 5))
sns.barplot(x='InternetService', y='Churn', data=df, errorbar=None)
plt.title('Average Churn Rate By Internet Service Type')
plt.xlabel('Internet Service')
plt.ylabel('Mean Churn (1=yes)')
plt.xticks(rotation=45) # rotate labels for readability
plt.show()

# Visual 2: correlation heatmap with annotations
plt.figure(figsize=(10, 8))
corr_matrix = df.select_dtypes(include=['float', 'int']).corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of numerical Features')
plt.show()

# Visual 3: Boxplot for monthlycharges by churn with customizations
plt.figure(figsize=(8, 5))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, palette='Set3')# color palette for distinction
plt.title('Distributon of Monthly Charges by Churn Status')
plt.xlabel('Churn (0=No, 1=Yes)')
plt.ylabel('Monthly Charges')
plt.show() # churnes have higher median



# Findings and Recommendations
## Insights.
    . Churn is higher for fiber optic users(possibly due to higher cost); seniors and month-to-month customers are at risk.
    . Lower tenure(<10 months) and high monthly charges (>70) correlate with churn.
    . Add-ons like TechSupport reduce churn (lower rates for 'yes')

# Recommendations
    . Offer discounts for long-term contracts; bundle security / support for high-risk groups.
    . Target retention campaigns at new customers and fiber users.
    . Monitor charges-cap or explain high bills to prevent churn.



# APPENDIX
## .Reproducibility: Github workbrian369-ux.
## .References: IBM Telco Dataset; Pandas/seaborn.
